<a href="https://colab.research.google.com/github/Kubojah-Dan/kuboja-codeboosters-2026/blob/main/DAY7/Embeddings%2BSemantic_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Embeddings** **+** **Semantic** **Search**

In [2]:
!pip install chromadb sentence-transformers -q

print("Installation Complete")

Installation Complete


In [3]:
 #=====================================
 # Step 2: Import All Libraries
 #=====================================

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
print("All Libraries Imported Successfully")
print(f"ChromaDB Version: {chromadb.__version__}")


All Libraries Imported Successfully
ChromaDB Version: 1.5.9


In [5]:
#============================================
# Keyword Search vs Semantic Search
#============================================


# KEYWORD SEARCH
documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobiles",
    "SQL is used to query databases",
    "Machine learning trains models on data"
]

query_keyword = "vehicle"

print("*" * 50)
print(f"KEYWORD SEARCH for: {query_keyword}")
print("*" * 50)

for i, doc in enumerate(documents):
  if query_keyword.lower() in doc.lower():
    print(f" FOUND [doc_{i}] : {doc}")
  else:
    print(f" MISSED [doc_{i}] : {doc}")

print()

**************************************************
KEYWORD SEARCH for: vehicle
**************************************************
 MISSED [doc_0] : ETL is used to clean and transform data
 FOUND [doc_1] : A vehicle is a mode of transportation
 MISSED [doc_2] : Cars and trucks are popular automobiles
 MISSED [doc_3] : SQL is used to query databases
 MISSED [doc_4] : Machine learning trains models on data



In [9]:
# SEMANTIC SEARCH

print("Loading embedding model........")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding Model Loaded Successfully!")
print(f"Model process vector of size: {model.get_embedding_dimension()}")

Loading embedding model........


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded Successfully!
Model process vector of size: 384


In [11]:
# Generating Embeddings

sentence = "ETL is used to clean and transform data"

embedding = model.encode(sentence)

print(f"Input sentence: {sentence}")
print()
print(f"Embedding of sentence: {embedding}")
print(f"Embedding type: {type(embedding)}")
print(f"Embedding shape: {embedding.shape}")

print(f"First 10 numbers: {embedding[:10].round(4)}")
print(f"Min value: {embedding.min():.4f}")
print(f"Max value: {embedding.max():.4f}")

Input sentence: ETL is used to clean and transform data

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (384,)
First 10 numbers: [-0.0784  0.0541  0.0224 -0.0389  0.0221 -0.0904  0.0007 -0.0152  0.0733
  0.0362]
Min value: -0.1381
Max value: 0.1815


In [18]:
import numpy as np

documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobiles",
    "SQL is used to query databases",
    "Machine learning trains models on data"
]

query_text = "vehicle"

doc_embeddings = model.encode(documents)
query_embedding = model.encode(query_text)

print("*" * 50)
print(f"SEMANTIC SEARCH for: {query_text}")
print("*" * 50)

for i, doc in enumerate(documents):
  score = np.dot(query_embedding, doc_embeddings[i])
  if score > 0.4:
    print(f" FOUND [doc_{i}] : {doc}")
  else:
    print(f" MISSED [doc_{i}] : {doc}")

print()

**************************************************
SEMANTIC SEARCH for: vehicle
**************************************************
 MISSED [doc_0] : ETL is used to clean and transform data
 FOUND [doc_1] : A vehicle is a mode of transportation
 FOUND [doc_2] : Cars and trucks are popular automobiles
 MISSED [doc_3] : SQL is used to query databases
 MISSED [doc_4] : Machine learning trains models on data



In [21]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobiles",
    "SQL is used to query databases",
    "Machine learning trains models on data"
]
search_word = "facts"

doc_embeddings = model.encode(documents)
query_embedding = model.encode(search_word)

# Calculating similarities
similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]

print(f"SEMANTIC SEARCH (Manual) for: '{search_word}'\n")
results = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)

for doc, score in results:
    print(f"[Score: {score:.4f}] {doc}")

SEMANTIC SEARCH (Manual) for: 'facts'

[Score: 0.2448] Cars and trucks are popular automobiles
[Score: 0.1441] Machine learning trains models on data
[Score: 0.1254] SQL is used to query databases
[Score: 0.1107] ETL is used to clean and transform data
[Score: 0.0803] A vehicle is a mode of transportation


In [22]:


your_sentences = [
    "Machine learning trains models on labeled data",
    "AI algorithms learn patterns from examples",
    "I enjoy eating pizza for lunch"
]

your_embeddings = model.encode(your_sentences)
sin_your_01 = cosine_similarity(your_embeddings[0], your_embeddings[1])
sin_your_02 = cosine_similarity(your_embeddings[0], your_embeddings[2])

print(f"Similarity between '{your_sentences[0]}' and '{your_sentences[1]}': {sin_your_01:.4f}")
print(f"Similarity between '{your_sentences[0]}' and '{your_sentences[2]}': {sin_your_02:.4f}")

Similarity between 'Machine learning trains models on labeled data' and 'AI algorithms learn patterns from examples': 0.3293
Similarity between 'Machine learning trains models on labeled data' and 'I enjoy eating pizza for lunch': -0.0452


# **ChromaDB** **-** **Vector** **Database**

In [23]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection("demo_notes")

print("ChromaDB client created Successfuly")
print(f"Collection name: {collection}")
print(f"Documents in collection: {collection.count()}")

ChromaDB client created Successfuly
Collection name: Collection(name=demo_notes)
Documents in collection: 0


In [24]:
#=============================
# Adding Some Sample Documents
#=============================

sample_docs = [
    "ETL stands for Extract Transform Load - used to clean and transform data",
    "SQL SELECT statements retrieve data from database tables",
    "Machine learning models learn patterns from training data",
    "Python Pandas library is used for data manipulation and cleaning",
    "Neural networks are inspired by how the human brain works"
]

sample_ids = ["doc001", "doc002", "doc003", "doc004", "doc005"]

sample_metadata = [
    {"subject": "Data Engineering", "topic": "ETL"},
    {"subject": "Data Engineering", "topic": "SQL"},
    {"subject": "Machine Learning", "topic": "ML Basics"},
    {"subject": "Python", "topic": "Pandas"},
    {"subject": "Machine Learning", "topic": "Neural Networks"}
]

collection.add(
    documents=sample_docs,
    ids=sample_ids,
    metadatas=sample_metadata
)

print(f"Documents added to Collection")
print(f"Total documents: {collection.count()}")


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 49.6MiB/s]


Documents added to Collection
Total documents: 5


In [25]:

query = "How do I clean and prepare data?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print('RESULTS AVAILABLE:')
print(list(results.keys()))

RESULTS AVAILABLE:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [28]:
#========================================
# Printing Results in a Readable Format
#========================================

print(f"Query: '{query}'")
print()
print("*" * 50)
print()

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distances = results['distances'][0]
matched_metadata = results['metadatas'][0]

for rank, (doc, doc_id, dist, meta) in enumerate(zip(matched_docs, matched_ids, matched_distances, matched_metadata)):
  print(f"Rank: {rank} | ID: {doc_id} | Distances: {dist:.4f}")
  print(f" Subject: {meta['subject']} | Topic: {meta['topic']}")
  print(f" Document: {doc}")
  print()

Query: 'How do I clean and prepare data?'

**************************************************

Rank: 0 | ID: doc004 | Distances: 1.1045
 Subject: Python | Topic: Pandas
 Document: Python Pandas library is used for data manipulation and cleaning

Rank: 1 | ID: doc001 | Distances: 1.5864
 Subject: Data Engineering | Topic: ETL
 Document: ETL stands for Extract Transform Load - used to clean and transform data

Rank: 2 | ID: doc002 | Distances: 1.6139
 Subject: Data Engineering | Topic: SQL
 Document: SQL SELECT statements retrieve data from database tables



In [30]:
filtered_query = "How do computers learn from examples"

filtered_results = collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"subject": "Machine Learning"}
)

print(f"FILTERED QUERY: {filtered_query}")
print("Filter: Only Machine Learning documents")
print()
print("*" * 60)

for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]
), start=1):
    print(f"Rank: {rank} | ID: {doc_id} | Distances: {dist:.4f}")
    print(f" Subject: {meta['subject']} | Topic: {meta['topic']}")
    print(f" Document: {doc}")
    print()

FILTERED QUERY: How do computers learn from examples
Filter: Only Machine Learning documents

************************************************************
Rank: 1 | ID: doc002 | Distances: 0.9956
 Subject: Machine Learning | Topic: ML Basics
 Document: Machine learning models learn patterns from training data

Rank: 2 | ID: doc002 | Distances: 1.1305
 Subject: Machine Learning | Topic: Neural Networks
 Document: Neural networks are inspired by how the human brain works



In [31]:
print("DISTANCE TO SIMILARITY CONVERSION")


distances = [0.05, 0.28, 0.48, 0.65, 0.90]
interpretations = ["Near identical", "Very similar", "Related", "Somewhat related", "Not related"]

print(f"{'Distance':<15} {'Similarity':<15} {'Interpretation':<20}")
print("="*60)

for dist, interp in zip(distances, interpretations):
    # Convert distance to similarity score (assuming 0-1 range for distance)
    # A common way for normalized distances is: similarity = 1 - distance
    similarity = 1 - dist

    print(f"{dist:<15.2f} {similarity:<15.2f} {interp:<20}")

DISTANCE TO SIMILARITY CONVERSION
Distance        Similarity      Interpretation      
0.05            0.95            Near identical      
0.28            0.72            Very similar        
0.48            0.52            Related             
0.65            0.35            Somewhat related    
0.90            0.10            Not related         


In [32]:
notes_df = pd.read_csv("college_notes.csv")

print("Dataset loaded!")
print(f"Shape: {notes_df.shape}")
print(f"Columns: {list(notes_df.columns)}")
print(notes_df.head())

Dataset loaded!
Shape: (15, 4)
Columns: ['note_id', 'subject', 'topic', 'content']
  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N005  Data Engineering      Big Data and PySpark   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Big Data refers to extremely large datasets th...  


In [33]:
first_note = notes_df.iloc[1]

print(f"Note ID: {first_note['note_id']}")
print(f"Subject: {first_note['subject']}")
print(f"Topic: {first_note['topic']}")
print(f"Content: {first_note['content']}")

Note ID: N002
Subject: Data Engineering
Topic: SQL Databases
Content: A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.


In [36]:
all_ids = notes_df['note_id'].tolist()

all_metadata = [
    {'subject': row['subject'], 'topic': row['topic']}
    for _, row in notes_df.iterrows()
]

print(f"Documents prepared: {len(all_ids)}")
print(f"First 5 IDs: {all_ids[:5]}")
print(f"Metadata prepared: {len(all_metadata)}")
print()
print(f"Sample ID: {all_ids[0]}")
print(f"Sample Metadata: {all_metadata[0]}")
print("Sample document (first 80 chars):", notes_df.iloc[0]['content'][:80])

Documents prepared: 15
First 5 IDs: ['N001', 'N002', 'N003', 'N004', 'N005']
Metadata prepared: 15

Sample ID: N001
Sample Metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
Sample document (first 80 chars): ETL stands for Extract Transform Load. It is the process of collecting raw data 


In [35]:
#=========================================
# MINI PROJECT: Setup ChromaDB Collection
#=========================================

client = chromadb.EphemeralClient()

notes_collection = client.get_or_create_collection(
    name="college_notes"
)

print("ChromaDB client created successfully")
print(f"Collection name: {notes_collection}")
print(f"Documents in collection: {notes_collection.count()}")

ChromaDB client created successfully
Collection name: Collection(name=college_notes)
Documents in collection: 0


In [39]:
def show_results(query, results):
    print(f"Query: '{query}'")
    print("*" * 50)

    if not results['ids'] or not results['ids'][0]:
        print("No results found.")
        return

    matched_docs = results['documents'][0]
    matched_ids = results['ids'][0]
    matched_distances = results['distances'][0]
    matched_metadata = results['metadatas'][0]

    for rank, (doc, doc_id, dist, meta) in enumerate(zip(matched_docs, matched_ids, matched_distances, matched_metadata)):
        print(f"Rank: {rank} | ID: {doc_id} | Distance: {dist:.4f}")
        print(f" Subject: {meta['subject']} | Topic: {meta['topic']}")
        print(f" Document: {doc}")
        print()

notes_collection.add(
    ids=all_ids,
    metadatas=all_metadata,
    documents=notes_df['content'].tolist()
)

query_1 = "How do I fix messy and incomplete data?"
results_1 = notes_collection.query(
    query_texts=[query_1],
    n_results=3
)

show_results(query_1, results_1)

Query: 'How do I fix messy and incomplete data?'
**************************************************
Rank: 0 | ID: N003 | Distance: 0.8342
 Subject: Data Engineering | Topic: Data Cleaning
 Document: Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.

Rank: 1 | ID: N015 | Distance: 1.6552
 Subject: Python Programming | Topic: Data Visualization
 Document: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts line plots histograms and pie charts that help humans understand patterns in data.

Rank: 2 | ID: N014 | Distance: 1.6644
 Subject: Python Programming | Topic: Pandas Library
 Document: Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table